# Общий эффект добавления номера договора в ключ

Тетрадка считает:
- сколько уникальных карточек получается при базовом ключе `inn + fp_num + fp_type + IDENTIFICATION_DATE`;
- сколько уникальных карточек получится при добавлении `agreement_num` в ключ;
- разницу в абсолюте и процентах;
- среднее и медиану количества договоров на один ИНН.

Экспорт в Excel не выполняется (по умолчанию только вывод в тетрадке).

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

DATA_DIR = Path('/home/jovyan/documents/DMUKP_FP_DEF/data')
CRM_FILE = DATA_DIR / 'crm_last_version.csv'

DATE_FROM = pd.Timestamp('2023-01-01')
DATE_TO = pd.Timestamp('2025-12-31')

ALLOWED_SOURCES = ['Н2.0', 'Справочник CRM-системы', 'CRM-система']
SEGMENT_MAP = {
    'ДМСБ (микро)': 'МкБ',
    'ДМСБ (малый)': 'МСБ',
    'ДМСБ (средний)': 'МСБ',
    'ДКБ': 'ККБ',
}

BASE_KEY = ['inn', 'fp_num', 'fp_type', 'IDENTIFICATION_DATE']
AGREEMENT_COLS = ['AGREEMENT_NUM', 'AGREEMENT_NUM_1']


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith('.0'):
        s = s[:-2]
    s = s.lstrip('0') or '0'
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print('CRM_FILE:', CRM_FILE)

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
raw.columns = raw.columns.str.strip()

if 'VAL' in raw.columns:
    raw = raw[raw['VAL'].astype(str).str.strip().isin(ALLOWED_SOURCES)].copy()

raw['inn'] = raw['X_INN'].apply(normalize_inn)
raw['dt'] = pd.to_datetime(raw['IDENTIFICATION_DATE'], dayfirst=True, errors='coerce')
raw = raw[(raw['dt'] >= DATE_FROM) & (raw['dt'] <= DATE_TO)].copy()

raw['segment'] = raw['X_AREA_RESP'].astype(str).str.strip().map(SEGMENT_MAP)
raw = raw[raw['segment'].notna()].copy()

raw = raw.dropna(subset=['inn', 'NUMBER_FP_SFP']).copy()
raw['fp_num'] = raw['NUMBER_FP_SFP'].astype(str).str.strip()
raw['fp_type'] = raw['TYPE_FP'].astype(str).str.strip().replace({'FP': 'ФП', 'SFP': 'СФП'})

for col in AGREEMENT_COLS:
    work_col = f'agreement_{col.lower()}'
    if col in raw.columns:
        raw[work_col] = raw[col].astype(str).str.strip()
    else:
        raw[work_col] = ''
    raw.loc[raw[work_col] == '', work_col] = '[EMPTY_AGREEMENT]'

print('Rows after filters:', len(raw))
print('Unique INN:', raw['inn'].nunique())
for col in AGREEMENT_COLS:
    work_col = f'agreement_{col.lower()}'
    print(f"Rows with empty agreement replaced ({col}):", int((raw[work_col] == '[EMPTY_AGREEMENT]').sum()))

In [ ]:
cards_base_key = raw.drop_duplicates(BASE_KEY).shape[0]

summary_rows = []
for col in AGREEMENT_COLS:
    work_col = f'agreement_{col.lower()}'
    key_with_agreement = BASE_KEY + [work_col]
    cards_key_with_agreement = raw.drop_duplicates(key_with_agreement).shape[0]
    delta_cards = cards_key_with_agreement - cards_base_key
    delta_pct_vs_base = round((delta_cards / cards_base_key * 100), 4) if cards_base_key else None

    summary_rows.append(
        {
            'agreement_source': col,
            'cards_base_key': int(cards_base_key),
            'cards_key_with_agreement': int(cards_key_with_agreement),
            'delta_cards': int(delta_cards),
            'delta_pct_vs_base': delta_pct_vs_base,
        }
    )

summary_key_compare = pd.DataFrame(summary_rows)
display(summary_key_compare)

In [ ]:
distribution_rows = []
agreements_per_inn_all = []
value_counts_all = []

for col in AGREEMENT_COLS:
    work_col = f'agreement_{col.lower()}'
    agreements_per_inn = (
        raw.groupby('inn', as_index=False)
        .agg(agreements_per_inn=(work_col, pd.Series.nunique))
    )
    agreements_per_inn['agreement_source'] = col
    agreements_per_inn_all.append(agreements_per_inn)

    distribution_rows.append(
        {
            'agreement_source': col,
            'inn_count': int(len(agreements_per_inn)),
            'mean_agreements_per_inn': round(float(agreements_per_inn['agreements_per_inn'].mean()), 4) if len(agreements_per_inn) else None,
            'median_agreements_per_inn': float(agreements_per_inn['agreements_per_inn'].median()) if len(agreements_per_inn) else None,
            'p75': float(agreements_per_inn['agreements_per_inn'].quantile(0.75)) if len(agreements_per_inn) else None,
            'p90': float(agreements_per_inn['agreements_per_inn'].quantile(0.90)) if len(agreements_per_inn) else None,
            'max': int(agreements_per_inn['agreements_per_inn'].max()) if len(agreements_per_inn) else None,
            'inn_with_more_than_one_agreement': int((agreements_per_inn['agreements_per_inn'] > 1).sum()),
        }
    )

    vc = (
        agreements_per_inn['agreements_per_inn']
        .value_counts(dropna=False)
        .rename_axis('agreements_per_inn')
        .reset_index(name='inn_count')
        .sort_values('agreements_per_inn')
        .reset_index(drop=True)
    )
    vc['agreement_source'] = col
    value_counts_all.append(vc)

agreements_distribution_summary = pd.DataFrame(distribution_rows)
agreements_per_inn_value_counts = pd.concat(value_counts_all, ignore_index=True)
agreements_per_inn_all = pd.concat(agreements_per_inn_all, ignore_index=True)

print('Статистика количества договоров на ИНН (по обоим полям):')
display(agreements_distribution_summary)
print('Распределение agreements_per_inn:')
display(agreements_per_inn_value_counts.head(100))
print('TOP ИНН по количеству договоров (по каждому полю):')
display(agreements_per_inn_all.sort_values(['agreement_source', 'agreements_per_inn'], ascending=[True, False]).head(60))

In [ ]:
check_rows = []
control_rows = []

rows_before = int(len(raw))
rows_after_base = int(raw.drop_duplicates(BASE_KEY).shape[0])

for col in AGREEMENT_COLS:
    work_col = f'agreement_{col.lower()}'
    key_with_agreement = BASE_KEY + [work_col]
    rows_after_agreement = int(raw.drop_duplicates(key_with_agreement).shape[0])

    aggr = agreements_per_inn_all[agreements_per_inn_all['agreement_source'] == col]
    check_rows.append(
        {
            'agreement_source': col,
            'check_name': 'cards_key_with_agreement_ge_cards_base_key',
            'passed': bool(rows_after_agreement >= rows_after_base),
        }
    )
    check_rows.append(
        {
            'agreement_source': col,
            'check_name': 'all_agreements_per_inn_ge_1',
            'passed': bool((aggr['agreements_per_inn'] >= 1).all()) if len(aggr) else True,
        }
    )

    control_rows.extend([
        {'agreement_source': col, 'metric': 'rows_before', 'value': rows_before},
        {'agreement_source': col, 'metric': 'rows_after_base_key', 'value': rows_after_base},
        {'agreement_source': col, 'metric': 'rows_after_key_with_agreement', 'value': rows_after_agreement},
        {'agreement_source': col, 'metric': 'removed_by_base_key', 'value': rows_before - rows_after_base},
        {'agreement_source': col, 'metric': 'removed_by_key_with_agreement', 'value': rows_before - rows_after_agreement},
    ])

checks = pd.DataFrame(check_rows)
control_totals = pd.DataFrame(control_rows)

print('Контрольные проверки:')
display(checks)
print('Контрольные итоги:')
display(control_totals)